In [0]:

#business -> joins , aggregations + window functions and case 


In [0]:
results_df = spark.table("formula1_dev.silver.results")
races_df = spark.table("formula1_dev.silver.races")
drivers_df = spark.table("formula1_dev.silver.drivers")
constructors_df = spark.table("formula1_dev.silver.constructors")

In [0]:
from pyspark.sql.functions import col
drivers_race_df = results_df.alias("res").join(races_df.alias("rac"), col("res.race_id") == col("rac.race_id"), "inner").join(drivers_df.alias("dr"), col("res.driver_id") == col("dr.driver_id"), "inner")\
    .select(col("rac.race_year"),
            col("rac.round"),
            col("rac.race_id"),
            col("res.driver_id"),
            col("dr.driver_full_name"),
            col("res.position"),
            col("res.points"),
            col("res.laps")
            )
drivers_race_df.display()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *
race_window = Window.partitionBy("race_year").orderBy(desc(col("round")))
df_final = drivers_race_df.withColumn("race_number", row_number().over(race_window))\
    .withColumn("leading_position", lead("position").over(race_window))\
        .withColumn("lag_position", lag("position").over(race_window))
df_final.display()

In [0]:
drivers_race_df.display()

In [0]:
drivers_season_df = drivers_race_df.groupBy("race_year", "driver_id", "driver_full_name")\
    .agg(sum("points").alias("total_points"),
         countDistinct("race_id").alias("races_entered"),
         sum(when(col("position") == 1,1).otherwise(0)).alias("winner"),
         sum(when(col("position") == 2,1).otherwise(0)).alias("second"),
         sum(when(col("position") == 3,1).otherwise(0)).alias("third"))
drivers_season_df.display()

In [0]:
champion_window = Window.partitionBy("race_year").orderBy(desc(col("total_points")))
drivers_season_df_final= drivers_season_df.withColumn("rank", rank().over(champion_window))\
    .withColumn("rn", row_number().over(champion_window))\
        .withColumn("dn", dense_rank().over(champion_window))\
            .withColumn("ntile", ntile(4).over(champion_window))
drivers_season_df_final.display()
# 100 --> 1 --> 25 
#         2--> 25-50
#         3  --> 50-75
#         4 --> 75-100

In [0]:
drivers_season_df_final.write.format("delta").mode("overwrite").saveAsTable("formula1_dev.gold.championship_drivers")

In [0]:
testing --> bronze 
next power bi\data science --> 
dashboard --> 